In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

inputs = Path("/kaggle/input")
print("top-level /kaggle/input entries:", list(inputs.iterdir()))
roots = list((inputs / "nanowm-code").rglob("pyproject.toml"))
if not roots:
    roots = list(inputs.rglob("pyproject.toml"))
if len(roots) != 1:
    raise RuntimeError(f"Expected exactly one project root, found {len(roots)}: {roots}")
mounted = roots[0].parent
root = Path("/kaggle/working/project")
root.mkdir(parents=True, exist_ok=True)
shutil.copytree(mounted, root, dirs_exist_ok=True)
for archive in mounted.glob("*.zip"):
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root))
print("project root:", root)

In [ ]:
subprocess.run([sys.executable, "scripts/preflight_gpu.py"], check=True)
# transformers is what src/data/dinov2.py loads DINOv2-small through; it is
# usually preinstalled on Kaggle, pinned here so a missing dependency fails
# now rather than after the render pass has already spent quota.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], check=True)

In [ ]:
# Inference-side: Kaggle weekly quota, NOT the 20-hour training budget.
# Re-runs the M3 tier with DINOv2-small features alongside the AE latents.
# Without them the two REPA arms of the ablation cannot be built at all --
# TrajectoryWindowDataset(with_dinov2=True) raises rather than silently
# training REPA against nothing.
subprocess.run([
    sys.executable, "scripts/preprocess/precompute_dataset.py",
    "--tier", "m3",
    "--out-dir", "/kaggle/working/nanowm_data/m3",
    "--device", "cuda",
    "--with-dinov2",
], check=True)

In [ ]:
# Cheap guard before spending the ticket: confirm the features are really
# there and shaped as the loss expects, rather than discovering it after
# torch.compile has already run four times.
import numpy as np, json
manifest = json.load(open("/kaggle/working/nanowm_data/m3/manifest.json"))
print("dinov2_model_id:", manifest["dinov2_model_id"], "dim:", manifest["dinov2_feature_dim"])
sample = sorted(Path("/kaggle/working/nanowm_data/m3").glob("*.npz"))[0]
with np.load(sample) as data:
    print(sample.name, {k: data[k].shape for k in data.files})
    assert data["dinov2"].shape[0] == data["latents"].shape[0]
    assert data["dinov2"].shape[1] == data["latents"].shape[1] == 16
    assert data["dinov2"].shape[2] == 384

In [ ]:
# configs/m3_profile.yaml ships with ticket_hours: null and
# target_seconds_per_arm: null (tests/test_m3_profile.py guards both --
# never commit a live ticket value). The approved values are patched in
# only on this Kaggle copy.
#
# Ticket M3-A, approved 2026-09-05: 0.35 GPU-hours.
#   Phase 1: 4 arms x (30 warmup + 100 timed) steps at 15M
#   Phase 2: 3 Muon LRs + 1 AdamW reference x 600 steps
# Estimated ~15 min of real compute; the rest is margin for eight
# torch.compile invocations, whose cost the step timings deliberately
# exclude.
#
# target_seconds_per_arm=1800 is the M3-B plan it converts throughput for:
# 8 arms x 1800s = 4.0 GPU-hours, leaving ~0.65h of M3's 5.0h in reserve
# after this profile. It is only an arithmetic input here -- no part of
# M3-B is approved or started by this notebook.
import yaml

cfg_path = Path("configs/m3_profile.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["run"]["ticket_hours"] = 0.35
cfg["profile"]["target_seconds_per_arm"] = 1800
cfg_path.write_text(yaml.safe_dump(cfg))
print(cfg_path.read_text())

In [ ]:
# run_m3_profile.py enforces its own ticket (scripts.train.check_ticket)
# and writes one ledger entry in a finally block whatever happens. It
# snapshots results after every arm, so a deadline abort (exit code 2)
# still leaves everything measured so far on disk.
result = subprocess.run([
    sys.executable, "scripts/run_m3_profile.py",
    "--config", str(cfg_path),
])
print(f"profile exit code: {result.returncode}")
print(Path("budget/ledger.jsonl").read_text())

In [ ]:
import json
profile = json.load(open(cfg["output"]["path"]))
print(json.dumps({k: v for k, v in profile.items() if k != "config"}, indent=2))